# Chapter 10: Diffusion Models for Images
This notebook accompanies Chapter 10 of *Deep Learning with PyTorch (2nd Edition)* by Howard Huang.
We implement the complete generative diffusion pipeline from scratch:
- **2D Point Cloud Dataset from Official PyTorch Logo** (resizing 1000x1000 to 256x256 and contour extraction)
- **Forward Diffusion Markov Chain** and **Closed-Form Jump**
- **Sinusoidal Positional Embeddings** for coordinates and time
- **Denoising Multi-Layer Perceptron (MLP)**
- **Simplified MSE Loss ($L_{simple}$)** and Training Loop
- **DDPM Ancestral Reverse Sampling** reconstructing data from pure Gaussian noise

In [ ]:
import io
import math
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
import torch
import torch.nn as nn
import torch.nn.functional as F

# Configure device and deterministic seed
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__} | Active Device: {device}")

## 2. Generating the 2D Contour Dataset (PyTorch Logo PNG -> 256x256 -> Points)
We fetch the official PyTorch logo symbol from a transparent PNG URL (or local file), downscale it from 1000x1000 to 256x256, extract edge contours via Sobel/FIND_EDGES filtering, and normalize the points to $(0, 0)$ mean with unit variance.

In [ ]:
LOGO_URL = "https://res.cloudinary.com/startup-grind/image/upload/c_fill,w_500,h_500,g_center/c_fill,dpr_2.0,f_auto,g_center,q_auto:good/v1/gcs/platform-data-linuxhq/events/PyTorch_Symbol_01_OrangeOnTransparent_nUWxXkQ.png"
LOCAL_FALLBACK_PATH = "../../content/img/deep-learning-with-pytorch/pytorch_logo_symbol.png"

def load_pytorch_logo_points(source=LOGO_URL, num_points=3000, target_size=(256, 256)):
    """
    Downloads or reads PyTorch logo, resizes to target_size (256x256), extracts edge contours,
    and centers/normalizes to mean=0, std=1.
    """
    img = None
    if source.startswith("http://") or source.startswith("https://"):
        try:
            req = urllib.request.Request(source, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=10) as resp:
                img_bytes = resp.read()
            img = Image.open(io.BytesIO(img_bytes)).convert("RGBA")
            print("Successfully loaded logo from URL.")
        except Exception as e:
            print(f"URL load failed ({e}), attempting local fallback...")
    
    if img is None:
        import os
        for path in [source, LOCAL_FALLBACK_PATH, "pytorch_logo_symbol.png"]:
            if os.path.exists(path):
                img = Image.open(path).convert("RGBA")
                print(f"Loaded logo from local file: {path}")
                break
                
    if img is None:
        print("Generating synthetic parametric PyTorch logo fallback...")
        theta = np.linspace(0, 2 * np.pi, int(num_points * 0.65), endpoint=False)
        ring_x, ring_y = 1.8 * np.cos(theta), 1.8 * np.sin(theta)
        flame_pts = int(num_points * 0.35)
        flame_x = np.concatenate([np.linspace(-0.5, 0.5, flame_pts // 2), np.zeros(flame_pts - flame_pts // 2)])
        flame_y = np.concatenate([np.linspace(-1.0, 1.2, flame_pts // 2), np.linspace(0.2, 1.5, flame_pts - flame_pts // 2)])
        coords = np.stack([np.concatenate([ring_x, flame_x]), np.concatenate([ring_y, flame_y])], axis=1).astype(np.float32)
        coords -= coords.mean(axis=0, keepdims=True)
        coords /= coords.std()
        return torch.tensor(coords, dtype=torch.float32)

    # Resize from original (e.g. 1000x1000) to 256x256
    img = img.resize(target_size, Image.Resampling.LANCZOS)
    
    # Edge contour extraction via Sobel/FIND_EDGES
    gray = img.convert("L")
    edges = gray.filter(ImageFilter.FIND_EDGES)
    edge_arr = np.array(edges)
    y_idx, x_idx = np.where(edge_arr > 40)
    
    if len(x_idx) < 500:
        alpha = np.array(img)[:, :, 3]
        y_idx, x_idx = np.where(alpha > 50)
        
    total = len(x_idx)
    indices = np.random.choice(total, num_points, replace=(total < num_points))
    x_pts = x_idx[indices].astype(np.float32)
    y_pts = -y_idx[indices].astype(np.float32)  # Invert Y to Cartesian orientation
    
    coords = np.stack([x_pts, y_pts], axis=1)
    coords -= coords.mean(axis=0, keepdims=True)
    coords /= coords.std()
    return torch.tensor(coords, dtype=torch.float32)

x0 = load_pytorch_logo_points(LOGO_URL, num_points=3000, target_size=(256, 256))
print(f"x0 Dataset Shape: {x0.shape} | Mean: {x0.mean():.4f} | Std: {x0.std():.4f}")

plt.figure(figsize=(5, 5))
plt.scatter(x0[:, 0].numpy(), x0[:, 1].numpy(), s=3, c="#e94560", alpha=0.7)
plt.title("Pristine Points x0 (PyTorch Logo Extracted from PNG)")
plt.xlim(-2.5, 2.5)
plt.ylim(-2.5, 2.5)
plt.grid(True, alpha=0.3)
plt.show()

## 3. Linear Variance Schedule & Analytical Precomputations
We define a linear variance schedule $\beta_t \in [10^{-4}, 0.02]$ across $T = 1000$ steps and precompute $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$.

In [ ]:
def linear_beta_schedule(timesteps=1000, start=0.0001, end=0.02):
    return torch.linspace(start, end, timesteps)

T = 1000
betas = linear_beta_schedule(timesteps=T)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

print(f"Schedule: beta_0 = {betas[0]:.6f}, beta_{T-1} = {betas[-1]:.6f}")
print(f"Cumulative: alpha_bar_0 = {alphas_cumprod[0]:.6f}, alpha_bar_{T-1} = {alphas_cumprod[-1]:.6f}")

## 4. Closed-Form Forward Diffusion Jump
Instead of iterating step-by-step, we sample directly: $\mathbf{x}_t = \sqrt{\bar{\alpha}_t}\mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t}\boldsymbol{\epsilon}$, where $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$.

In [ ]:
def reshape_for_x(a, x):
    batch_size = x.shape[0]
    ones_to_broadcast = len(x.shape) - 1
    return a.view(batch_size, *([1] * ones_to_broadcast)).to(x.device)

def forward_diffusion_sample(x0_tensor, t_tensor, device=device):
    x0_tensor = x0_tensor.to(device)
    noise = torch.randn_like(x0_tensor)
    sqrt_alpha_bar = reshape_for_x(sqrt_alphas_cumprod[t_tensor], x0_tensor)
    sqrt_one_minus_alpha_bar = reshape_for_x(sqrt_one_minus_alphas_cumprod[t_tensor], x0_tensor)
    xt = sqrt_alpha_bar * x0_tensor + sqrt_one_minus_alpha_bar * noise
    return xt, noise

# Visualize the forward degradation progression
test_timesteps = [0, 100, 250, 500, 750, 999]
fig, axes = plt.subplots(1, len(test_timesteps), figsize=(18, 3))
for idx, t_step in enumerate(test_timesteps):
    t = torch.full((x0.shape[0],), t_step, dtype=torch.long)
    xt, _ = forward_diffusion_sample(x0, t, device="cpu")
    axes[idx].scatter(xt[:, 0], xt[:, 1], s=2, c="#0f3460", alpha=0.5)
    axes[idx].set_title(f"Forward Step t = {t_step}")
    axes[idx].set_xlim(-3, 3)
    axes[idx].set_ylim(-3, 3)
    axes[idx].axis("off")
plt.suptitle("Closed-Form Forward Diffusion Degradation")
plt.show()

## 5. Sinusoidal Positional Embeddings
Diffusion step indices $t$ are projected into dense high-frequency frequency vectors via harmonic sine-cosine curves.

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        
    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

pos_test = SinusoidalPositionEmbeddings(dim=64)
sample_t = torch.arange(0, 1000, 10)
sample_emb = pos_test(sample_t)
print(f"Embedding Matrix Shape: {sample_emb.shape}")

plt.figure(figsize=(8, 4))
plt.imshow(sample_emb.cpu().numpy(), aspect="auto", cmap="viridis")
plt.colorbar(label="Activation")
plt.xlabel("Embedding Dimension (0..63)")
plt.ylabel("Timestep index / 10")
plt.title("Sinusoidal Timestep Embeddings Across Range t in [0, 1000]")
plt.show()

## 6. Denoising Multi-Layer Perceptron (MLP)
A residual MLP architecture receiving noisy 2D points $\mathbf{x}_t$ conditioned on time embeddings $\mathbf{e}_t$.

In [ ]:
class DenoiseMLP(nn.Module):
    def __init__(self, hidden_dim=256, time_dim=64):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        self.input_layer = nn.Linear(2, hidden_dim)
        
        self.block1 = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        self.block2 = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        self.output_layer = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_dim, 2)
        )
        
    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        h = self.input_layer(x)
        h = h + t_emb
        h = h + self.block1(h)
        h = h + t_emb
        h = h + self.block2(h)
        return self.output_layer(h)

model = DenoiseMLP().to(device)
print(f"Total Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 7. Model Training Loop with Simplified Loss ($L_{simple}$)
We train the model by minimizing: $L_{simple} = \mathbb{E}_{t, \mathbf{x}_0, \boldsymbol{\epsilon}} [\Vert \boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t) \Vert^2]$

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 256
dataset = TensorDataset(x0)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
num_epochs = 40
loss_history = []

model.train()
for epoch in range(1, num_epochs + 1):
    epoch_losses = []
    for (batch_x,) in dataloader:
        batch_x = batch_x.to(device)
        optimizer.zero_grad()
        
        t = torch.randint(0, T, (batch_x.shape[0],), device=device).long()
        xt, noise = forward_diffusion_sample(batch_x, t, device=device)
        predicted_noise = model(xt, t)
        
        loss = F.mse_loss(predicted_noise, noise)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
        
    avg_loss = np.mean(epoch_losses)
    loss_history.append(avg_loss)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{num_epochs:02d}] | Denoising MSE Loss: {avg_loss:.5f}")

plt.figure(figsize=(6, 3))
plt.plot(loss_history, color="#e94560", lw=2)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("DDPM Denoising Training Loss Convergence")
plt.grid(True, alpha=0.3)
plt.show()

## 8. DDPM Ancestral Reverse Sampling
Generating brand-new pristine coordinates from pure Gaussian noise $\mathbf{x}_T \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ by stepping backwards from $T-1$ to 0.

In [ ]:
@torch.no_grad()
def sample_timestep(model_net, x, t):
    betas_t = reshape_for_x(betas[t], x)
    sqrt_one_minus_alpha_bar_t = reshape_for_x(sqrt_one_minus_alphas_cumprod[t], x)
    sqrt_recip_alpha_t = reshape_for_x(sqrt_recip_alphas[t], x)
    
    model_mean = sqrt_recip_alpha_t * (
        x - (betas_t * model_net(x, t) / sqrt_one_minus_alpha_bar_t)
    )
    
    if t[0] == 0:
        return model_mean
    else:
        posterior_var_t = reshape_for_x(posterior_variance[t], x)
        z = torch.randn_like(x)
        return model_mean + torch.sqrt(posterior_var_t) * z

@torch.no_grad()
def generate_reverse_trajectory(model_net, num_points=2500):
    model_net.eval()
    current_x = torch.randn(num_points, 2, device=device)
    
    save_steps = [999, 750, 500, 250, 100, 0]
    trajectory = {}
    
    for step in reversed(range(T)):
        t = torch.full((num_points,), step, dtype=torch.long, device=device)
        current_x = sample_timestep(model_net, current_x, t)
        if step in save_steps:
            trajectory[step] = current_x.cpu().numpy()
            
    return current_x.cpu().numpy(), trajectory

final_pts, traj = generate_reverse_trajectory(model)

## 9. Visualizing Generative Reconstruction: Chaos to Structure
We observe how the model progressively eliminates ambiguity and resurrects the PyTorch logo.

In [ ]:
vis_timesteps = [999, 750, 500, 250, 100, 0]
fig, axes = plt.subplots(1, len(vis_timesteps), figsize=(18, 3))
for idx, t_val in enumerate(vis_timesteps):
    pts = traj[t_val]
    axes[idx].scatter(pts[:, 0], pts[:, 1], s=2, c="#ffaa00" if t_val > 0 else "#52b788", alpha=0.6)
    axes[idx].set_title(f"Reverse Step t = {t_val}")
    axes[idx].set_xlim(-2.5, 2.5)
    axes[idx].set_ylim(-2.5, 2.5)
    axes[idx].axis("off")
plt.suptitle("Generative Reverse Reconstruction: Chaos to PyTorch Logo")
plt.show()